In [4]:
import time
import random
import hashlib
import pandas as pd
from collections import deque
# import ace_tools as tools; # Removed ace_tools import

# Helper functions for Bloom Filter
class BloomFilter:
    def __init__(self, size, hash_count):
        self.size = size
        self.hash_count = hash_count
        self.bit_array = [0] * size

    def _hashes(self, key):
        for i in range(self.hash_count):
            digest = hashlib.md5(f"{key}-{i}".encode()).hexdigest()
            yield int(digest, 16) % self.size

    def add(self, key):
        for idx in self._hashes(key):
            self.bit_array[idx] = 1

    def __contains__(self, key):
        return all(self.bit_array[idx] for idx in self._hashes(key))

# IBLT implementation with efficient decode
class IBLTCell:
    def __init__(self):
        self.count = 0
        self.key_xor = 0
        self.hash_xor = 0

class IBLT:
    def __init__(self, m, hash_count):
        self.m = m
        self.hash_count = hash_count
        self.table = [IBLTCell() for _ in range(m)]
        self.key_map = {}  # key_int -> original key

    def _hashes(self, key):
        for i in range(self.hash_count):
            digest = hashlib.sha1(f"{key}-{i}".encode()).hexdigest()
            yield int(digest, 16) % self.m

    def _to_int(self, key):
        return int(hashlib.sha256(str(key).encode()).hexdigest(), 16)

    def insert(self, key):
        key_int = self._to_int(key)
        self.key_map[key_int] = key
        for idx in self._hashes(key):
            cell = self.table[idx]
            cell.count += 1
            cell.key_xor ^= key_int
            cell.hash_xor ^= key_int

    def delete(self, key):
        key_int = self._to_int(key)
        for idx in self._hashes(key):
            cell = self.table[idx]
            cell.count -= 1
            cell.key_xor ^= key_int
            cell.hash_xor ^= key_int

    def list_entries(self):
        queue = deque(i for i, cell in enumerate(self.table) if abs(cell.count) == 1)
        entries = []  # (key, sign)
        while queue:
            idx = queue.popleft()
            cell = self.table[idx]
            if abs(cell.count) != 1:
                continue
            sign = cell.count
            key_int = cell.key_xor
            original_key = self.key_map.get(key_int)
            entries.append((original_key, sign))
            # remove from table
            for h in self._hashes(original_key):
                c = self.table[h]
                c.count -= sign
                c.key_xor ^= key_int
                c.hash_xor ^= key_int
                if abs(c.count) == 1:
                    queue.append(h)
        # verify emptiness
        if any(cell.count != 0 or cell.key_xor != 0 or cell.hash_xor != 0 for cell in self.table):
            return None
        return entries

# Simulation parameters
n = 100000
diff = 10000

# Generate test sets
local_keys = list(range(n))
extra_keys = random.sample(range(n * 2), diff)
remote_keys = local_keys + extra_keys

# 1. Bloom Filter test
bf_size = n * 20
bf_hashes = 7
bf = BloomFilter(bf_size, bf_hashes)

start = time.time()
for key in local_keys:
    bf.add(key)
bf_build_time = time.time() - start

start = time.time()
false_positives = sum(1 for key in extra_keys if key in bf)
bf_test_time = time.time() - start

# 2. IBLT test
iblt_size = diff * 4
iblt_hashes = 3
iblt = IBLT(iblt_size, iblt_hashes)

start = time.time()
for key in remote_keys:
    iblt.insert(key)
for key in local_keys:
    iblt.delete(key)
iblt_build_time = time.time() - start

start = time.time()
decoded_entries = iblt.list_entries()
iblt_decode_time = time.time() - start

# Evaluate decode success (only positive sign entries)
decoded_positive = {k for k, s in decoded_entries or [] if s == 1} if decoded_entries else set()
decode_success = decoded_positive == set(extra_keys)

# Compile results
results = pd.DataFrame([
    {
        "Data Structure": "Bloom Filter",
        "Build Time (s)": bf_build_time,
        "Test Time (s)": bf_test_time,
        "False Positives": false_positives,
        "Decode Success": None
    },
    {
        "Data Structure": "IBLT",
        "Build Time (s)": iblt_build_time,
        "Decode Time (s)": iblt_decode_time,
        "False Positives": None,
        "Decode Success": decode_success
    }
])

# tools.display_dataframe_to_user(name="Comparison of Bloom Filter vs IBLT", dataframe=results) # Removed ace_tools display
display(results) # Added standard display

,Data Structure,Build Time (s),Test Time (s),False Positives,Decode Success,Decode Time (s)
0,Bloom Filter,0.965842,0.052138,5023.0,None,NaN
1,IBLT,1.283760,NaN,NaN,True,0.050086
